In [1]:
import numpy as np
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [2]:
# Define classes and their labels
classes = {'NORMAL': 0, 'PNEUMONIA': 1}

# Load and preprocess the dataset
X = []
Y = []

for cls, label in classes.items():
    pth = os.path.join("..", "archive", "chest_xray", "train", cls)

    for filename in os.listdir(pth):
        img = cv2.imread(os.path.join(pth, filename), 0)

        img = cv2.resize(img, (128, 128))

        X.append(img.flatten() / 255.0)

        Y.append(label)

X = np.array(X)
Y = np.array(Y)

In [3]:
print(X.shape)
print(Y.shape)

(5216, 16384)
(5216,)


In [ ]:
# ============================================
# LOAD FIXED TRAINING DATASET
# ============================================

classes = {'NORMAL': 0, 'PNEUMONIA': 1}
xtrain = []
ytrain = []

for cls, label in classes.items():
    train_path = os.path.join(
        "..",
        "archive",
        "chest_xray",
        "train",
        cls
    )
    for filename in os.listdir(train_path):
        img = cv2.imread(
            os.path.join(train_path, filename),
            0
        )
        img = cv2.resize(img, (128, 128))
        xtrain.append(img.flatten() / 255.0)
        ytrain.append(label)

xtrain = np.array(xtrain)
ytrain = np.array(ytrain)

# ============================================
# LOAD FIXED TEST DATASET
# ============================================

xtest = []
ytest = []

for cls, label in classes.items():
    test_path = os.path.join(
        "..",
        "archive",
        "chest_xray",
        "test",
        cls
    )
    for filename in os.listdir(test_path):
        img = cv2.imread(
            os.path.join(test_path, filename),
            0
        )
        img = cv2.resize(img, (128, 128))
        xtest.append(img.flatten() / 255.0)
        ytest.append(label)

xtest = np.array(xtest)
ytest = np.array(ytest)

print("Training samples:", len(xtrain))
print("Testing samples:", len(xtest))
print("====================================")

# ============================================
# TRAINING PERCENTAGE EXPERIMENT
# ============================================

training_sizes = [0.2, 0.4, 0.6, 0.8]
rbf_scores = []
linear_scores = []

for size in training_sizes:
    x_small, _, y_small, _ = train_test_split(
        xtrain,
        ytrain,
        train_size=size,
        random_state=10
    )

    # PCA
    pca = PCA(n_components=0.98)
    pca_train = pca.fit_transform(x_small)
    pca_test = pca.transform(xtest)

    # ========================================
    # RBF KERNEL
    # ========================================

    rbf = SVC(kernel='rbf')
    rbf.fit(pca_train, y_small)
    rbf_train_acc = rbf.score(
        pca_train,
        y_small
    )
    rbf_test_acc = rbf.score(
        pca_test,
        ytest
    )
    rbf_scores.append(rbf_test_acc)

    # ========================================
    # LINEAR KERNEL
    # ========================================

    linear = SVC(kernel='linear')
    linear.fit(
        pca_train,
        y_small
    )
    linear_train_acc = linear.score(
        pca_train,
        y_small
    )
    linear_test_acc = linear.score(
        pca_test,
        ytest
    )
    linear_scores.append(linear_test_acc)

    print("Training size:", size)
    print("Samples:", len(x_small))
    print(
        "RBF Training Accuracy:",
        round(rbf_train_acc, 4)
    )
    print(
        "RBF Testing Accuracy:",
        round(rbf_test_acc, 4)
    )
    print(
        "Linear Training Accuracy:",
        round(linear_train_acc, 4)
    )
    print(
        "Linear Testing Accuracy:",
        round(linear_test_acc, 4)
    )
    print("--------------------------------")

# So later prediction cells work
sv = rbf

## RBF Kernel Scores

In [ ]:
print("RBF Training Accuracy:",
      round(rbf_train_acc,4))

print("RBF Testing Accuracy:",
      round(rbf_test_acc,4))

## Linear Kernel Scores

In [ ]:
print("Linear Training Accuracy:",
      round(linear_train_acc,4))

print("Linear Testing Accuracy:",
      round(linear_test_acc,4))

In [ ]:
# Display sample images and predictions

def display_samples(folder, title, num_samples=9):
    plt.figure(figsize=(12, 8))
    c = 1
    for i, filename in enumerate(os.listdir(folder)[:num_samples]):
        img = cv2.imread(
            os.path.join(folder, filename),
            0
        )

        img = cv2.resize(img, (128, 128))
        flat_img = img.flatten() / 255.0
        prediction = sv.predict(
            pca.transform([flat_img])
        )

        plt.subplot(3, 3, c)
        plt.title(dec[prediction[0]])
        plt.imshow(img, cmap='gray')
        plt.axis('off')
        c += 1
    plt.suptitle(title)
    plt.show()

In [ ]:
# Define class labels

dec = {
    0: 'NORMAL',
    1: 'PNEUMONIA'
}

# Display sample images and predictions

xray_folders = ['NORMAL', 'PNEUMONIA']
for xray_folder in xray_folders:
    folder_path = os.path.join(
        "..",
        "archive",
        "chest_xray",
        "test",
        xray_folder
    )
    title = f'{xray_folder} Samples'
    display_samples(folder_path, title)

In [ ]:
# Count the occurrences of each class

xray_counts = {
    dec[label]: np.sum(ytrain == label)
    for label in classes.values()
}

# Create table

table_data = {
    'Index': range(1, len(classes) + 1),
    'Class': list(xray_counts.keys()),
    'Count': list(xray_counts.values())
}
xray_table = pd.DataFrame(table_data)

# Style table

styled_table = xray_table.style.set_properties(
    **{
        'border': '2px solid black',
        'text-align': 'center'
    }
)
print("\033[1mChest X-Ray Class Counts:\033[0m")
display(styled_table)

In [ ]:
# Histogram of class distribution

xray_labels = [dec[label] for label in ytrain]
colors = ['skyblue', 'orange']
plt.figure(figsize=(8, 6))
for label, color in zip(classes.values(), colors):
    plt.hist(
        np.array(xray_labels)[ytrain == label],
        bins=len(classes),
        align='mid',
        rwidth=0.8,
        color=color,
        label=dec[label]
    )
plt.xlabel('Classes')
plt.ylabel('Number of Samples')
plt.title('Distribution of Chest X-Ray Classes')
plt.xticks(
    range(len(classes)),
    list(classes.keys())
)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

percentages = [20,40,60,80]
plt.figure(figsize=(8,5))

plt.plot(
    percentages,
    rbf_scores,
    marker='o',
    label='RBF Kernel'
)

plt.plot(
    percentages,
    linear_scores,
    marker='o',
    label='Linear Kernel'
)

plt.xlabel("Training Data Percentage")
plt.ylabel("Testing Accuracy")
plt.title(
    "Training Data Percentage vs Accuracy"
)
plt.legend()
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
import numpy as np

# Use the final trained linear model
y_pred = sv.predict(pca_test)

# Accuracy
accuracy = sv.score(pca_test, ytest)

# F1 Score
f1 = f1_score(
    ytest,
    y_pred,
    average='binary'
)

# Sensitivity (Recall)
sensitivity = recall_score(
    ytest,
    y_pred,
    average='binary'
)

# Confusion matrix
cm = confusion_matrix(
    ytest,
    y_pred
)

# Specificity calculation
specificity_list=[]

for i in range(len(cm)):
    TP=cm[i,i]
    FP=np.sum(cm[:,i])-TP
    FN=np.sum(cm[i,:])-TP
    TN=np.sum(cm)-TP-FP-FN
    specificity=TN/(TN+FP)
    specificity_list.append(specificity)

specificity=np.mean(specificity_list)

print("Accuracy:",round(accuracy,4))
print("F1 Score:",round(f1,4))
print("Sensitivity:",round(sensitivity,4))
print("Specificity:",round(specificity,4))